In [ ]:
import os
import gc
import torch
import numpy as np
import cv2
from PIL import Image
from tqdm.notebook import tqdm
from diffusers import StableDiffusionXLControlNetInpaintPipeline, ControlNetModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_DIR = "/content/drive/MyDrive/cleaned/test" # Manually change the path to each dataset directory
MASK_DIR = "/content/drive/MyDrive/cleaned/test/masks"

OUT_DIR_RUSTY = "/content/drive/MyDrive/cleaned/test/outputs_rusty"
OUT_DIR_ITASHA = "/content/drive/MyDrive/cleaned/test/outputs_itasha"
OUT_DIR_FROZEN = "/content/drive/MyDrive/cleaned/test/outputs_frozen"

for d in [OUT_DIR_RUSTY, OUT_DIR_ITASHA, OUT_DIR_FROZEN]:
    os.makedirs(d, exist_ok=True)

STYLE_CONFIGS = {
    "rusty": {"output_dir": OUT_DIR_RUSTY, "prompt": "abandoned rusty car, heavily damaged, peeling paint, oxidized metal, dents, post-apocalyptic, highly detailed, realistic textures", "negative_prompt": "clean, new, shiny, perfect, smooth, showroom, pristine", "strength": 0.85, "control_scale": 0.85},
    "itasha": {"output_dir": OUT_DIR_ITASHA, "prompt": "car fully wrapped in vibrant anime itasha graphics, colorful Japanese manga character decals, vector art livery, highly detailed custom paint job, glossy finish", "negative_prompt": "rusty, dirty, old, deformed, plain color, solid red, text, watermark", "strength": 0.95, "control_scale": 0.85},
    "frozen": {"output_dir": OUT_DIR_FROZEN, "prompt": "car completely covered in thick ice and snow, frozen solid, icicles hanging, frost on surface, winter blizzard environment, highly detailed, realistic", "negative_prompt": "warm, summer, dry, sunny, fire, melting, clean, clear glass", "strength": 0.80, "control_scale": 0.80}
}

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=torch.float16, use_safetensors=True
).to(DEVICE)

pipe = StableDiffusionXLControlNetInpaintPipeline.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    controlnet=controlnet, torch_dtype=torch.float16, variant="fp16"
).to(DEVICE)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)

def get_canny_image(image_pil):
    img_np = np.array(image_pil)
    canny = cv2.Canny(img_np, 100, 200)
    canny = canny[:, :, None]
    return Image.fromarray(np.concatenate([canny, canny, canny], axis=2))

def resize_for_sdxl_safety(img, max_size=1024):
    w, h = img.size
    scale = 1.0
    if max(w, h) > max_size:
        scale = max_size / float(max(w, h))

    new_w = int(w * scale)
    new_h = int(h * scale)
    new_w = (new_w // 8) * 8
    new_h = (new_h // 8) * 8

    if (new_w, new_h) != (w, h):
        return img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    return img

mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('.png')]

for mask_filename in tqdm(mask_files, desc="Batch Generation"):
    base_name = os.path.splitext(mask_filename)[0]

    img_path_jpg = os.path.join(IMAGE_DIR, base_name + ".jpg")
    img_path_png = os.path.join(IMAGE_DIR, base_name + ".png")

    if os.path.exists(img_path_jpg): 
        img_path, ext = img_path_jpg, ".jpg"
    elif os.path.exists(img_path_png): 
        img_path, ext = img_path_png, ".png"
    else: 
        continue

    try:
        init_image_raw = Image.open(img_path).convert("RGB")
        mask_image_raw = Image.open(os.path.join(MASK_DIR, mask_filename)).convert("RGB")

        original_size = init_image_raw.size

        safe_init_image = resize_for_sdxl_safety(init_image_raw)
        safe_mask_image = resize_for_sdxl_safety(mask_image_raw)
        safe_canny_image = get_canny_image(safe_init_image)

    except Exception as e:
        tqdm.write(f"I/O Error - {base_name}: {e}")
        continue

    for style_name, config in STYLE_CONFIGS.items():
        out_filename = base_name + f"_{style_name}" + ext
        out_path = os.path.join(config["output_dir"], out_filename)

        if os.path.exists(out_path):
            continue

        try:
            output = pipe(
                prompt=config["prompt"],
                negative_prompt=config["negative_prompt"],
                image=safe_init_image,
                mask_image=safe_mask_image,
                control_image=safe_canny_image,
                num_inference_steps=35,
                strength=config["strength"],
                controlnet_conditioning_scale=config["control_scale"],
                guidance_scale=8.0
            ).images[0]

            if output.size != original_size:
                output = output.resize(original_size, Image.Resampling.LANCZOS)

            output.save(out_path)
        except Exception as e:
            tqdm.write(f"Generation Error - {base_name} - {style_name}: {e}")

    gc.collect()
    torch.cuda.empty_cache()